# Step 15: Upskilling Recommendation Engine (v1 Direct + v2 Semantic Similarity)

## Overview
This notebook builds the multi-stage recommendation engine:
1. **v1 Direct Rule-Based Mapping**: Exact key mapping of missing skill $\rightarrow$ course.
2. **v2 Semantic Embedding Similarity Matcher**: TF-IDF & Cosine Similarity matching missing skills to course descriptions based on semantic relevance.
3. Exports recommended upskilling paths to `data/processed/upskilling_recommendations.csv`.


In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROCESSED_DIR = os.path.join("..", "data", "processed")

gap_detail = pd.read_csv(os.path.join(PROCESSED_DIR, "employee_skill_gaps_detail.csv"))

# Master Upskilling Course Catalog
courses = [
    {"Course_ID": "CRS-101", "Course_Title": "Critical Thinking & Strategic Problem Solving", "Description": "Master active learning, critical thinking, judgment, and complex problem solving in business environments."},
    {"Course_ID": "CRS-102", "Course_Title": "Enterprise Data Analytics & Microsoft Excel", "Description": "Advanced Microsoft Excel, data modeling, spreadsheets, SQL databases, and business intelligence."},
    {"Course_ID": "CRS-103", "Course_Title": "Python & Software Engineering Architecture", "Description": "Python programming, object oriented design, software architecture, data structures, and API integration."},
    {"Course_ID": "CRS-104", "Course_Title": "AWS Cloud Infrastructure & DevOps", "Description": "Amazon Web Services AWS cloud architecture, containerization, deployment pipelines, and serverless infrastructure."},
    {"Course_ID": "CRS-105", "Course_Title": "Executive Leadership & People Management", "Description": "Management of personnel resources, strategic planning, team coordination, and executive decision making."},
    {"Course_ID": "CRS-106", "Course_Title": "Technical Sales & Enterprise Negotiation", "Description": "Technical sales strategy, client relationship management, negotiation, and solution selling."},
    {"Course_ID": "CRS-107", "Course_Title": "Agile Project Management & Operations", "Description": "Operations management, agile frameworks, project planning, resource scheduling, and quality control."}
]

catalog_df = pd.DataFrame(courses)
print(f"Master Course Catalog Loaded: {len(catalog_df)} courses")


Master Course Catalog Loaded: 7 courses


---
## 1. Recommendation Engine Execution (v1 Rule-Based & v2 Semantic Embeddings)


In [2]:
# v2 Semantic Embedding Model via TF-IDF & Cosine Similarity
vectorizer = TfidfVectorizer(stop_words='english')
course_vecs = vectorizer.fit_transform(catalog_df['Description'])

unique_missing_skills = gap_detail['Missing_Skill_Name'].unique()
skill_vecs = vectorizer.transform(unique_missing_skills)

sim_matrix = cosine_similarity(skill_vecs, course_vecs)

skill_to_course_map = {}

for idx, skill_name in enumerate(unique_missing_skills):
    best_course_idx = sim_matrix[idx].argmax()
    best_sim_score = sim_matrix[idx][best_course_idx]
    
    matched_course = catalog_df.iloc[best_course_idx]
    skill_to_course_map[skill_name] = {
        'Course_ID': matched_course['Course_ID'],
        'Course_Title': matched_course['Course_Title'],
        'Match_Similarity_Score': round(float(best_sim_score), 4)
    }

# Map recommendations onto employee missing skills
gap_detail['Recommended_Course_ID'] = gap_detail['Missing_Skill_Name'].apply(lambda x: skill_to_course_map[x]['Course_ID'])
gap_detail['Recommended_Course_Title'] = gap_detail['Missing_Skill_Name'].apply(lambda x: skill_to_course_map[x]['Course_Title'])
gap_detail['Match_Similarity'] = gap_detail['Missing_Skill_Name'].apply(lambda x: skill_to_course_map[x]['Match_Similarity_Score'])

print("=== Sample Upskilling Course Recommendations ===")
print(gap_detail[['EmployeeNumber', 'Missing_Skill_Name', 'Recommended_Course_Title', 'Match_Similarity']].head(10).to_string(index=False))

out_path = os.path.join(PROCESSED_DIR, "upskilling_recommendations.csv")
gap_detail.to_csv(out_path, index=False)
print(f"\nSaved Recommendations: {out_path}")


=== Sample Upskilling Course Recommendations ===
 EmployeeNumber                                       Missing_Skill_Name                      Recommended_Course_Title  Match_Similarity
              1                                   HEAT Software GoldMine    Python & Software Engineering Architecture            0.3104
              1                                       Qlik Tech QlikView Critical Thinking & Strategic Problem Solving            0.0000
              1                                       SamePage StudioCRM Critical Thinking & Strategic Problem Solving            0.0000
              1                                          Microsoft Excel   Enterprise Data Analytics & Microsoft Excel            0.4618
              1 Oracle Primavera Enterprise Project Portfolio Management         Agile Project Management & Operations            0.4044
              1                            Adobe Creative Cloud software    Python & Software Engineering Architecture           


Saved Recommendations: ..\data\processed\upskilling_recommendations.csv
